# Cafe Sales - Data Cleaning

Cleaning the `dirty_cafe_sales.csv` dataset (10,000 rows). The raw data has missing values, and some columns use `"ERROR"` / `"UNKNOWN"` as placeholder text instead of real values.

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

## Load the data

In [2]:
df = pd.read_csv('dirty_cafe_sales.csv')
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


## First look at the data

In [3]:
df.shape

(10000, 8)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [5]:
df.isna().sum()

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(0)

No fully duplicated rows. Every column is being read as text right now, even the numeric and date ones, so that needs to be fixed next.

## Fix column types

Quantity, Price Per Unit and Total Spent should be numbers, and Transaction Date should be a date. Any value that can't be converted (like the `"ERROR"` text) becomes `NaN` automatically.

In [7]:
for col in ['Quantity', 'Price Per Unit', 'Total Spent']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], format='%Y-%m-%d', errors='coerce')

df.dtypes

Transaction ID                 str
Item                           str
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
dtype: object

## Clean column names

In [8]:
df.columns = df.columns.str.strip().str.title()
df = df.rename(columns={'Transaction Id': 'Transaction ID'})
df.columns

Index(['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent',
       'Payment Method', 'Location', 'Transaction Date'],
      dtype='str')

## Treat "ERROR" and "UNKNOWN" as missing values

These show up as text in `Item`, `Payment Method` and `Location`. They're not real data, so it's easier to convert them to `NaN` now and deal with all missing values the same way.

In [9]:
df[['Item', 'Payment Method', 'Location']] = df[['Item', 'Payment Method', 'Location']].replace(['ERROR', 'UNKNOWN'], np.nan)

df.isna().sum()

Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

## Fill missing Item using Price Per Unit

Each menu item has a fixed price, so a known price tells us what the item most likely is. A couple of prices ($3 and $4) are shared by two items, so those get picked randomly between the two.

In [10]:
price_to_item = {
    1:   ['Cookie'],
    1.5: ['Tea'],
    2:   ['Coffee'],
    3:   ['Cake', 'Juice'],
    4:   ['Sandwich', 'Smoothie'],
    5:   ['Salad'],
}

def fill_item_from_price(df):
    for price, items in price_to_item.items():
        mask = (df['Price Per Unit'] == price) & (df['Item'].isna())
        if len(items) == 1:
            df.loc[mask, 'Item'] = items[0]
        else:
            df.loc[mask, 'Item'] = np.random.choice(items, size=mask.sum())
    return df

df = fill_item_from_price(df)
df['Item'].isna().sum()

np.int64(54)

## Fill missing Price Per Unit using Item

Same idea in reverse: if the item is known, its price is known too.

In [11]:
item_to_price = {
    'Cookie': 1, 'Tea': 1.5, 'Coffee': 2,
    'Cake': 3, 'Juice': 3,
    'Sandwich': 4, 'Smoothie': 4,
    'Salad': 5,
}

df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Item'].map(item_to_price))
df['Price Per Unit'].isna().sum()

np.int64(54)

## Fill the rest of Price Per Unit from Total Spent and Quantity

For the few rows where Item is also missing, Price Per Unit can still be recovered as `Total Spent / Quantity`.

In [12]:
df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Total Spent'] / df['Quantity'])
df['Price Per Unit'].isna().sum()

np.int64(6)

## Second pass on Item

More prices are known now, so running the same Item-filling step again catches a few more rows.

In [13]:
df = fill_item_from_price(df)
df['Item'].isna().sum()

np.int64(6)

## Fill missing Quantity and Total Spent

Both can be calculated from the other two columns.

In [14]:
df['Quantity'] = df['Quantity'].fillna(df['Total Spent'] / df['Price Per Unit'])
df['Total Spent'] = df['Total Spent'].fillna(df['Price Per Unit'] * df['Quantity'])

df[['Quantity', 'Total Spent']].isna().sum()

Quantity       23
Total Spent    23
dtype: int64

## Check Transaction ID for duplicates

This column should be unique for every row.

In [15]:
df['Transaction ID'].duplicated().sum()

np.int64(0)

## Payment Method and Location

About a third of these are missing, and there's no other column that can tell us what they should be. Filling them with the most common value would badly distort the real distribution, so they're labeled `"Unknown"` instead - that keeps them honest about being missing rather than guessed.

In [16]:
df['Payment Method'] = df['Payment Method'].fillna('Unknown')
df['Location'] = df['Location'].fillna('Unknown')

df[['Payment Method', 'Location']].isna().sum()

Payment Method    0
Location          0
dtype: int64

## Drop rows that can't be recovered

A small number of rows are still missing Item, Quantity, Price Per Unit or Total Spent, with nothing left to calculate them from. These rows can't be used for any revenue analysis, so they get dropped. The number is tiny compared to the full dataset, so this has no real effect on the results.

In [17]:
before = df.shape[0]

df = df.dropna(subset=['Item', 'Quantity', 'Price Per Unit', 'Total Spent'])
df = df.reset_index(drop=True)

after = df.shape[0]
print(f"Dropped {before - after} rows ({(before - after) / before * 100:.2f}%)")

Dropped 26 rows (0.26%)


## Transaction Date

Transaction Date is left as missing (`NaT`) for rows where it couldn't be read. Every other column in those rows is still valid, so they're kept - they just get excluded from any analysis that's specifically based on date.

## Final check

In [18]:
df.isna().sum()

Transaction ID        0
Item                  0
Quantity              0
Price Per Unit        0
Total Spent           0
Payment Method        0
Location              0
Transaction Date    460
dtype: int64

In [19]:
df.shape

(9974, 8)

## Save the cleaned dataset

In [20]:
df.to_csv('cleaned_cafe_sales.csv', index=False)